# Mid-SNR Expert — Task 5: Teacher trên noise sạch

Huấn luyện teacher BEATs trên **waveform noise sạch** (`noise_path`), rồi precompute
bank embedding + logits cho **toàn bộ** hàng manifest.

Teacher là *privileged information* (Lopez-Paz et al., ICLR 2016): nó chỉ tồn tại lúc
train. Lúc infer chỉ còn student nhìn mixture.

**Chốt dừng ở cuối notebook.** Acc teacher < 0.75 ⇒ dừng hẳn, đừng chạy Task 6:
teacher nhìn noise sạch mà không giỏi hơn baseline nhìn mixture bao nhiêu thì toàn bộ
tiền đề của hướng này sụp.

Tiến độ tổng thể ở `STATUS.md`. Lý do thiết kế ở `DESIGN.md`.

## 1. CONFIG — mọi tham số nằm trong một cell

In [ ]:
CONFIG = {
    "seed": 2026,

    # Đường dẫn. Chỉnh REPO/DATA cho đúng layout molab rồi chạy.
    "repo_beats": "../BEATs_Experts",          # tương đối so với SNR_Aware/Mid_Expert/
    "data_root": "/marimo/dataset/mix-dataset",
    "out_dir": "artifacts",

    # Checkpoint pretrained. *.pt bị .gitignore chặn nên KHÔNG đi theo git clone.
    # Phải có sẵn trên molab. Thử lần lượt các đường dẫn dưới đây.
    "pretrained_candidates": [
        "../BEATs_Experts/checkpoint/pretrained/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt",
        "../../BEATs/C_Noise_Separation_Fusion/checkpoint/checkpoint4/pretrained/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt",
        "/marimo/checkpoint/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt",
    ],

    "audio": {"sample_rate": 16000, "seconds": 4.0, "num_samples": 64000},
    "mid_band_db": [5.0, 10.0],

    "teacher": {
        # Task 1 đã xác minh ratio noise_path/rows = 1.0 ⇒ KHÔNG dedup.
        # Mỗi hàng train có file noise riêng; dedup theo noise_ytid sẽ sai vì
        # cắt mất các đoạn khác nhau của cùng nguồn.
        "dedup": False,

        "head_epochs": 50, "head_lr": 1e-3, "head_batch_size": 512, "head_patience": 10,
        "extract_batch_size": 64,

        "finetune_epochs": 8, "trainable_blocks": 12,
        "head_ft_lr": 1e-4, "encoder_lr": 1e-5,
        "batch_size": 32, "validation_batch_size": 64,
        "accumulation_steps": 1, "grad_clip": 5.0, "patience": 3,

        "workers": 4,
        "checkpoint": "teacher_noise_best.pt",
        "bank": "teacher_bank.pt",
        "history": "teacher_history.json",
    },

    # Chốt dừng (DESIGN.md §10)
    "gates": {"teacher_acc_floor": 0.75, "teacher_acc_ceiling": 0.95},

    "smoke_test": False,   # True => 400 clip, 1 epoch, để kiểm tra pipeline chạy thông
}

import json as _json
print(_json.dumps(CONFIG, indent=2, ensure_ascii=False))

## 2. Bootstrap — import lại code trong repo, không viết lại

In [ ]:
import sys, os, json, time, random, gc
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

REPO = Path(CONFIG["repo_beats"]).resolve()
DATA = Path(CONFIG["data_root"])
OUT = Path(CONFIG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# Dùng lại hàm sẵn có thay vì viết lại.
from noise_pipeline.mix_data import load_float_audio, load_mix_manifest
from train_beats_head import load_beats, initialize_head, metrics

SEED = CONFIG["seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

print("repo :", REPO)
print("data :", DATA, "exists:", DATA.exists())
print("out  :", OUT.resolve())
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

In [ ]:
# Tìm checkpoint pretrained. Dừng ngay và nói rõ nếu không có — *.pt không đi theo git.
PRETRAINED = None
for candidate in CONFIG["pretrained_candidates"]:
    p = Path(candidate)
    if p.exists():
        PRETRAINED = p.resolve(); break
    print("  khong thay:", p)

if PRETRAINED is None:
    raise FileNotFoundError(
        "Khong tim thay checkpoint pretrained BEATs.\n"
        "*.pt bi .gitignore chan nen KHONG di theo git clone — file nay phai co san\n"
        "tren molab. Them duong dan dung vao CONFIG['pretrained_candidates'] roi chay lai."
    )
print("pretrained:", PRETRAINED, f"({PRETRAINED.stat().st_size/1e6:.0f} MB)")

## 3. Audit manifest

Đã chạy local trên `36_labels/` và cho kết quả dưới đây. Chạy lại ở đây để xác nhận
bản trên molab giống hệt — nếu lệch thì phần còn lại của notebook không đáng tin.

| Kiểm tra | Kỳ vọng |
|---|---|
| tổng số hàng | 43.200 (train 30.240 / val 6.480 / test 6.480) |
| `unique noise_path / train rows` | **1.0** ⇒ không dedup |
| `mixture == clean + noise` | sai số < 1e-6 |
| SNR đo được vs `target_snr_db` | lệch 0 |

In [ ]:
ROWS = load_mix_manifest(DATA)
LABELS = (DATA / "labels.txt").read_text(encoding="utf-8").splitlines()
LABEL_TO_INDEX = {name: i for i, name in enumerate(LABELS)}

by_split = {}
for r in ROWS:
    by_split.setdefault(r["split"], []).append(r)

train_rows = by_split["train"]
ratio = len(train_rows) / len({r["noise_path"] for r in train_rows})

print("rows      :", len(ROWS), {k: len(v) for k, v in by_split.items()})
print("labels    :", len(LABELS))
print("columns   :", sorted(ROWS[0].keys()))
print("reuse ratio noise_path:", round(ratio, 4))

assert len(LABELS) == 36, f"expected 36 labels, got {len(LABELS)}"
if abs(ratio - 1.0) > 0.01:
    print(f"\n!! CANH BAO: ratio={ratio:.3f}, khac 1.0 nhu audit local.")
    print("   Xem lai quyet dinh 'khong dedup' truoc khi train teacher.")
else:
    print("OK - moi hang co file noise rieng, khong dedup (dung nhu audit local).")

In [ ]:
# Xác nhận mixture = clean + noise và SNR khớp manifest, trên 200 clip.
rng = random.Random(SEED)
sample = rng.sample(train_rows, min(200, len(train_rows)))
N = CONFIG["audio"]["num_samples"]; SR = CONFIG["audio"]["sample_rate"]

errs, devs = [], []
has_clean = "clean_path" in ROWS[0]
for r in sample:
    mix = load_float_audio(DATA / r["mixture_path"], SR, N)
    noi = load_float_audio(DATA / r["noise_path"], SR, N)
    spe = load_float_audio(DATA / r["clean_path"], SR, N) if has_clean else mix - noi
    errs.append(float((mix - (spe + noi)).abs().max()))
    snr = 10 * torch.log10(spe.pow(2).mean() / (noi.pow(2).mean() + 1e-20))
    devs.append(float(snr) - float(r["target_snr_db"]))

errs, devs = np.array(errs), np.array(devs)
print(f"clean_path co san : {has_clean}")
print(f"|mix-(clean+noise)| max = {errs.max():.3e}")
print(f"SNR deviation      mean={devs.mean():+.4f}  absmax={np.abs(devs).max():.4f}")

LINEAR_OK = bool(errs.max() < 1e-6 and np.abs(devs).max() < 1.0)
print("LINEAR_OK =", LINEAR_OK, "=> Task 8 (remix)", "chay duoc" if LINEAR_OK else "BO")

## 4. Dataset noise sạch

In [ ]:
class NoiseOnlyDataset(Dataset):
    """Waveform noise sạch + nhãn. Đầu vào của teacher ở giai đoạn 1.

    Trả cả row_index để bank embedding căn đúng thứ tự hàng manifest — student
    tra bank bằng chính chỉ số này, lệch một hàng là CRD học trên cặp sai mà
    không hề báo lỗi.
    """

    def __init__(self, root, rows, row_index, label_to_index, num_samples, sample_rate):
        self.root = Path(root)
        self.rows = rows
        self.row_index = row_index
        self.label_to_index = label_to_index
        self.n = num_samples
        self.sr = sample_rate

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        return {
            "audio": load_float_audio(self.root / row["noise_path"], self.sr, self.n),
            "target": self.label_to_index[row["label_names"]],
            "snr": int(float(row["target_snr_db"])),
            "row_index": self.row_index[i],
        }


GLOBAL_INDEX = {id(r): i for i, r in enumerate(ROWS)}

def make_noise_dataset(rows):
    return NoiseOnlyDataset(
        DATA, rows, [GLOBAL_INDEX[id(r)] for r in rows], LABEL_TO_INDEX,
        CONFIG["audio"]["num_samples"], CONFIG["audio"]["sample_rate"],
    )

teacher_train_rows = by_split["train"]
teacher_val_rows = by_split["validation"]

if CONFIG["smoke_test"]:
    teacher_train_rows = teacher_train_rows[:400]
    teacher_val_rows = teacher_val_rows[:200]
    print("!! SMOKE TEST — ket qua khong dung de bao cao")

print("teacher train:", len(teacher_train_rows), " val:", len(teacher_val_rows))

## 5. Giai đoạn A — head trên embedding đóng băng

Trích embedding BEATs một lần rồi train head tuyến tính trên cache. Rẻ hơn nhiều so với
finetune, và head được **warm-start từ hàng predictor AudioSet** tương ứng 36 nhãn
(`initialize_head`) chứ không khởi tạo ngẫu nhiên.

In [ ]:
def autocast_ctx():
    return torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.bfloat16,
        enabled=DEVICE.type == "cuda" and torch.cuda.is_bf16_supported(),
    )


@torch.inference_mode()
def extract_embeddings(model, rows, tag):
    loader = DataLoader(
        make_noise_dataset(rows),
        batch_size=CONFIG["teacher"]["extract_batch_size"],
        num_workers=CONFIG["teacher"]["workers"],
        shuffle=False, pin_memory=DEVICE.type == "cuda",
    )
    xs, ys, ss, ix = [], [], [], []
    t0 = time.perf_counter()
    for bi, batch in enumerate(loader, 1):
        with autocast_ctx():
            seq, _ = model.extract_features(batch["audio"].to(DEVICE, non_blocking=True))
        xs.append(seq.mean(dim=1).float().cpu())
        ys.append(batch["target"].long()); ss.append(batch["snr"].long())
        ix.append(batch["row_index"].long())
        if bi == 1 or bi % 100 == 0 or bi == len(loader):
            print(f"  extract {tag} {bi}/{len(loader)}", flush=True)
    return {"x": torch.cat(xs), "y": torch.cat(ys), "snr": torch.cat(ss),
            "row_index": torch.cat(ix), "seconds": time.perf_counter() - t0}


beats, raw_ckpt = load_beats(DEVICE, PRETRAINED)
print("BEATs loaded, encoder layers:", len(beats.encoder.layers))

cache_path = OUT / ("teacher_emb_smoke.pt" if CONFIG["smoke_test"] else "teacher_emb.pt")
if cache_path.exists():
    cached = torch.load(cache_path, map_location="cpu", weights_only=True)
    train_emb, val_emb = cached["train"], cached["validation"]
    print("dung cache:", cache_path)
else:
    train_emb = extract_embeddings(beats, teacher_train_rows, "train")
    val_emb = extract_embeddings(beats, teacher_val_rows, "validation")
    torch.save({"train": train_emb, "validation": val_emb}, cache_path)
    print(f"cached -> {cache_path}")

print("train emb:", tuple(train_emb["x"].shape), f"{train_emb['seconds']:.0f}s")
print("val   emb:", tuple(val_emb["x"].shape))

In [ ]:
label_to_mid = {}
for r in ROWS:
    label_to_mid.setdefault(r["label_names"], r["label_mids"])

head = initialize_head(raw_ckpt, LABELS, label_to_mid, DEVICE)
print("head warm-started tu predictor AudioSet:", tuple(head.weight.shape))


@torch.inference_mode()
def head_logits(h, data, batch=2048):
    return torch.cat([h(data["x"][i:i+batch].to(DEVICE)).cpu()
                      for i in range(0, len(data["x"]), batch)])


def train_head_stage(h, tr, va, epochs, lr, batch_size, patience):
    loader = DataLoader(TensorDataset(tr["x"], tr["y"]), batch_size=batch_size,
                        shuffle=True, generator=torch.Generator().manual_seed(SEED))
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(h.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5,
                                                       patience=2, min_lr=1e-6)
    best = metrics(head_logits(h, va), va["y"], va["snr"], LABELS)
    best_state = {k: v.detach().cpu().clone() for k, v in h.state_dict().items()}
    stale, hist = 0, []
    print(f"epoch=0 val_acc={best['accuracy']:.4f} val_f1={best['macro_f1']:.4f}")
    for ep in range(1, epochs + 1):
        h.train(); tot = cor = seen = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = h(xb); loss = crit(logits, yb)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += loss.item() * yb.shape[0]
            cor += (logits.argmax(1) == yb).sum().item(); seen += yb.shape[0]
        h.eval()
        m = metrics(head_logits(h, va), va["y"], va["snr"], LABELS)
        sched.step(m["macro_f1"])
        hist.append({"epoch": ep, "train_loss": tot/seen, "train_acc": cor/seen,
                     "val_acc": m["accuracy"], "val_macro_f1": m["macro_f1"]})
        print(f"epoch={ep} loss={tot/seen:.4f} train_acc={cor/seen:.4f} "
              f"val_acc={m['accuracy']:.4f} val_f1={m['macro_f1']:.4f}", flush=True)
        if m["macro_f1"] > best["macro_f1"] + 1e-4:
            best, stale = m, 0
            best_state = {k: v.detach().cpu().clone() for k, v in h.state_dict().items()}
        else:
            stale += 1
            if stale >= patience:
                print("early_stop", ep); break
    h.load_state_dict(best_state)
    return best, hist


tcfg = CONFIG["teacher"]
head_best, head_hist = train_head_stage(
    head, train_emb, val_emb,
    1 if CONFIG["smoke_test"] else tcfg["head_epochs"],
    tcfg["head_lr"], tcfg["head_batch_size"], tcfg["head_patience"],
)
print(f"\nHEAD  val_acc={head_best['accuracy']:.4f} val_macro_f1={head_best['macro_f1']:.4f}")

del train_emb, val_emb
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 6. Giai đoạn B — finetune 12 block cuối

Chỉ các block được mở khoá mới bật dropout; block đóng băng giữ nguyên chế độ eval để
kết quả xác định.

In [ ]:
for p in beats.parameters():
    p.requires_grad = False
for layer in beats.encoder.layers[-tcfg["trainable_blocks"]:]:
    for p in layer.parameters():
        p.requires_grad = True

def set_train_mode():
    beats.eval()
    for layer in beats.encoder.layers[-tcfg["trainable_blocks"]:]:
        layer.train()
    head.train()


def make_loader(rows, batch_size, shuffle):
    return DataLoader(
        make_noise_dataset(rows), batch_size=batch_size, shuffle=shuffle,
        generator=torch.Generator().manual_seed(SEED) if shuffle else None,
        num_workers=tcfg["workers"], pin_memory=DEVICE.type == "cuda",
    )


@torch.inference_mode()
def evaluate(loader):
    beats.eval(); head.eval()
    L, Y, S = [], [], []
    for batch in loader:
        with autocast_ctx():
            seq, _ = beats.extract_features(batch["audio"].to(DEVICE, non_blocking=True))
            logits = head(seq.mean(dim=1))
        L.append(logits.float().cpu()); Y.append(batch["target"]); S.append(batch["snr"])
    return metrics(torch.cat(L), torch.cat(Y), torch.cat(S), LABELS)


train_loader = make_loader(teacher_train_rows, tcfg["batch_size"], True)
val_loader = make_loader(teacher_val_rows, tcfg["validation_batch_size"], False)

opt = torch.optim.AdamW(
    [{"params": [p for p in beats.parameters() if p.requires_grad], "lr": tcfg["encoder_lr"]},
     {"params": head.parameters(), "lr": tcfg["head_ft_lr"]}],
    weight_decay=1e-4,
)
crit = nn.CrossEntropyLoss()
trainable = [p for p in list(beats.parameters()) + list(head.parameters()) if p.requires_grad]
print("trainable tensors:", len(trainable))

best = evaluate(val_loader)
best_state = {"encoder": {k: v.detach().cpu().clone() for k, v in beats.state_dict().items()
                          if k.startswith("encoder.layers.")},
              "head": {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}}
print(f"epoch=0 val_acc={best['accuracy']:.4f} val_f1={best['macro_f1']:.4f}")

ft_hist, stale = [], 0
for ep in range(1, (1 if CONFIG["smoke_test"] else tcfg["finetune_epochs"]) + 1):
    set_train_mode()
    opt.zero_grad(set_to_none=True)
    tot = cor = seen = 0
    for bi, batch in enumerate(train_loader, 1):
        wav = batch["audio"].to(DEVICE, non_blocking=True)
        yb = batch["target"].to(DEVICE, non_blocking=True)
        with autocast_ctx():
            seq, _ = beats.extract_features(wav)
            logits = head(seq.mean(dim=1))
            loss = crit(logits, yb)
        if not torch.isfinite(loss):
            raise FloatingPointError("loss khong huu han — dung lai truoc khi luu checkpoint")
        (loss / tcfg["accumulation_steps"]).backward()
        if bi % tcfg["accumulation_steps"] == 0 or bi == len(train_loader):
            nn.utils.clip_grad_norm_(trainable, tcfg["grad_clip"])
            opt.step(); opt.zero_grad(set_to_none=True)
        tot += loss.item() * yb.shape[0]
        cor += (logits.argmax(1) == yb).sum().item(); seen += yb.shape[0]
        if bi == 1 or bi % 200 == 0 or bi == len(train_loader):
            vram = torch.cuda.max_memory_allocated()/1024**3 if DEVICE.type == "cuda" else 0
            print(f"  ep{ep} batch={bi}/{len(train_loader)} loss={tot/seen:.4f} "
                  f"acc={cor/seen:.4f} vram={vram:.1f}GB", flush=True)
    m = evaluate(val_loader)
    ft_hist.append({"epoch": ep, "train_loss": tot/seen, "train_acc": cor/seen,
                    "val_acc": m["accuracy"], "val_macro_f1": m["macro_f1"]})
    print(f"epoch={ep} val_acc={m['accuracy']:.4f} val_f1={m['macro_f1']:.4f}", flush=True)
    if m["macro_f1"] > best["macro_f1"] + 1e-4:
        best, stale = m, 0
        best_state = {"encoder": {k: v.detach().cpu().clone() for k, v in beats.state_dict().items()
                                  if k.startswith("encoder.layers.")},
                      "head": {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}}
    else:
        stale += 1
        if stale >= tcfg["patience"]:
            print("early_stop", ep); break

beats.load_state_dict(best_state["encoder"], strict=False)
head.load_state_dict(best_state["head"])
torch.save({**best_state, "labels": LABELS, "validation_metrics": best,
            "pretrained": str(PRETRAINED)}, OUT / tcfg["checkpoint"])
(OUT / tcfg["history"]).write_text(
    json.dumps({"head_stage": head_hist, "finetune_stage": ft_hist,
                "best_validation": best}, indent=2), encoding="utf-8")
print(f"\nTEACHER val_acc={best['accuracy']:.4f} val_macro_f1={best['macro_f1']:.4f}")
print("checkpoint ->", OUT / tcfg["checkpoint"])

## 7. Chốt dừng — acc teacher

Con số này là **trần trên** của cả hướng tiếp cận. Ghi vào `STATUS.md`.

In [ ]:
acc = best["accuracy"]
floor, ceiling = CONFIG["gates"]["teacher_acc_floor"], CONFIG["gates"]["teacher_acc_ceiling"]
BASELINE_MID_ACC = 0.6681

print(f"teacher val accuracy = {acc:.4f}")
print(f"baseline BEATs tren lat mid (mixture) = {BASELINE_MID_ACC:.4f}")
print(f"khoang cach = {acc - BASELINE_MID_ACC:+.4f}\n")

if acc < floor:
    print("!! CHOT DUNG: acc < %.2f" % floor)
    print("   Teacher nhin noise SACH ma van khong gioi hon baseline nhin mixture bao nhieu.")
    print("   Tien de privileged-information sup. DUNG LAI, dung chay Task 6.")
    print("   Bao cao con so nay va hoi truoc khi di tiep.")
    TEACHER_GATE = "STOP"
elif acc > ceiling:
    print("!! CANH BAO: acc > %.2f" % ceiling)
    print("   Soft label gan one-hot => KD khong tai duoc thong tin gi.")
    print("   Dat rho_kd = 8.0 o Task 7 va ghi ro ly do trong bao cao.")
    TEACHER_GATE = "RAISE_RHO"
else:
    print("OK - teacher nam trong khoang lam viec duoc. Chay tiep sang buoc 8.")
    TEACHER_GATE = "PASS"

print("\nTEACHER_GATE =", TEACHER_GATE)

## 8. Bank embedding teacher

Chạy trên **toàn bộ 43.200 hàng** (train + validation + test), **không dedup**, giữ đúng
thứ tự hàng của `manifest.csv`. Student tra bank bằng `row_index`.

Teacher đóng băng sau bước này nên bank là tĩnh và chính xác — không phải memory buffer
cuốn chiếu như CRD gốc.

In [ ]:
@torch.inference_mode()
def build_bank(rows):
    loader = DataLoader(
        make_noise_dataset(rows), batch_size=tcfg["validation_batch_size"],
        num_workers=tcfg["workers"], shuffle=False, pin_memory=DEVICE.type == "cuda",
    )
    zs, lg, ys, ss, ix = [], [], [], [], []
    for bi, batch in enumerate(loader, 1):
        with autocast_ctx():
            seq, _ = beats.extract_features(batch["audio"].to(DEVICE, non_blocking=True))
            z = seq.mean(dim=1).float()
            logits = head(z)
        zs.append(z.half().cpu()); lg.append(logits.float().half().cpu())
        ys.append(batch["target"].long()); ss.append(batch["snr"].long())
        ix.append(batch["row_index"].long())
        if bi == 1 or bi % 100 == 0 or bi == len(loader):
            print(f"  bank {bi}/{len(loader)}", flush=True)
    return {"z": torch.cat(zs), "logits": torch.cat(lg), "y": torch.cat(ys),
            "snr": torch.cat(ss), "row_index": torch.cat(ix),
            "sample_id": [r["sample_id"] for r in rows]}


beats.eval(); head.eval()
bank_rows = ROWS[:800] if CONFIG["smoke_test"] else ROWS
bank = build_bank(bank_rows)
torch.save(bank, OUT / tcfg["bank"])
print("bank ->", OUT / tcfg["bank"],
      f"({(bank['z'].numel()*2 + bank['logits'].numel()*2)/1e6:.0f} MB)")
print("z:", tuple(bank["z"].shape), " logits:", tuple(bank["logits"].shape))

In [ ]:
# Xác minh bank căn đúng hàng. Lệch một hàng thì CRD học trên cặp sai mà KHÔNG báo lỗi —
# kết quả chỉ tồi đi và ta se do oan cho phuong phap.
expected_y = torch.tensor([LABEL_TO_INDEX[r["label_names"]] for r in bank_rows])
expected_snr = torch.tensor([int(float(r["target_snr_db"])) for r in bank_rows])

assert bank["z"].shape[0] == len(bank_rows), "so hang bank khac so hang manifest"
assert torch.equal(bank["row_index"], torch.arange(len(bank_rows))), "row_index khong tang dan"
assert torch.equal(bank["y"], expected_y), "nhan trong bank lech so voi manifest"
assert torch.equal(bank["snr"], expected_snr), "snr trong bank lech so voi manifest"
assert bank["sample_id"] == [r["sample_id"] for r in bank_rows], "sample_id lech"
assert torch.isfinite(bank["z"].float()).all(), "bank chua NaN/Inf"

print("BANK OK - can dung hang, khong NaN")
print("\n--- DAN VAO STATUS.md ---")
print(json.dumps({
    "teacher_val_accuracy": round(float(best["accuracy"]), 4),
    "teacher_val_macro_f1": round(float(best["macro_f1"]), 4),
    "teacher_gate": TEACHER_GATE,
    "bank_rows": int(bank["z"].shape[0]),
    "linear_ok": LINEAR_OK,
    "reuse_ratio": round(ratio, 4),
}, indent=2))